## Setup and Installations

In [1]:
!pip install transformers datasets evaluate rouge_score accelerate -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 14.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000

## Import Libraries and Configuration

In [2]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers import pipeline
import evaluate

# Configuration class to hold all parameters
class CFG:
    # Model name from Hugging Face
    MODEL_NAME = 't5-base'
    
    # Kaggle dataset path
    DATASET_PATH = '/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/'
    TRAIN_FILE = DATASET_PATH + 'train.csv'
    VAL_FILE = DATASET_PATH + 'validation.csv'
    TEST_FILE = DATASET_PATH + 'test.csv'
    
    # --- IMPORTANT ---
    # Set samples to None to run on the full dataset.
    # This will time out in a standard Kaggle notebook.
    N_TRAIN_SAMPLES = 50000  # Number of training samples (e.g., 10000 for a demo)
    N_VAL_SAMPLES = 5000     # Number of validation samples
    N_TEST_SAMPLES = 1000    # Number of test samples
    
    # Tokenizer settings
    MAX_INPUT_LENGTH = 512  # Max length of the input article
    MAX_TARGET_LENGTH = 128   # Max length of the output summary
    PREFIX = "summarize: "    # T5 models were trained with task-specific prefixes
    
    # Training parameters
    BATCH_SIZE = 8
    NUM_EPOCHS = 5
    LEARNING_RATE = 2e-5
    OUTPUT_DIR = "t5-summarizer-with-checkpoints"
    print("--- Configration Complete ---")


2025-10-26 15:11:53.080099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761491513.271816      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761491513.335047      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


--- Configration Complete ---


## Load and Clean Data

In [3]:
print("--- Starting Step 3: Load and Clean Data ---")

# Load the datasets
try:
    df_train = pd.read_csv(CFG.TRAIN_FILE)
    df_val = pd.read_csv(CFG.VAL_FILE)
    df_test = pd.read_csv(CFG.TEST_FILE)
    print(f"Original train shape: {df_train.shape}")
    print(f"Original val shape: {df_val.shape}")
    print(f"Original test shape: {df_test.shape}\n")
except FileNotFoundError:
    print(f"Error: Dataset files not found at {CFG.DATASET_PATH}")
    print("Please ensure the Kaggle dataset is added and the path is correct.")
    # Stop execution if files aren't found
    raise

# --- 1. Handle Nulls ---
print("Checking for null values...")
df_train = df_train.dropna(subset=['article', 'highlights'])
df_val = df_val.dropna(subset=['article', 'highlights'])
df_test = df_test.dropna(subset=['article', 'highlights'])
print(f"Shape after dropping nulls (train): {df_train.shape}\n")

# --- 2. Handle Duplicates ---
print("Checking for duplicate articles...")
df_train = df_train.drop_duplicates(subset=['article'], keep='first')
df_val = df_val.drop_duplicates(subset=['article'], keep='first')
df_test = df_test.drop_duplicates(subset=['article'], keep='first')
print(f"Shape after dropping duplicates (train): {df_train.shape}\n")

# --- 3. Subsample for Demo ---
if CFG.N_TRAIN_SAMPLES:
    df_train = df_train.sample(n=min(CFG.N_TRAIN_SAMPLES, len(df_train)), random_state=42)
if CFG.N_VAL_SAMPLES:
    df_val = df_val.sample(n=min(CFG.N_VAL_SAMPLES, len(df_val)), random_state=42)
if CFG.N_TEST_SAMPLES:
    df_test = df_test.sample(n=min(CFG.N_TEST_SAMPLES, len(df_test)), random_state=42)

print("--- Data Loading and Cleaning Complete ---")
print(f"Final training samples: {len(df_train)}")
print(f"Final validation samples: {len(df_val)}")
print(f"Final test samples: {len(df_test)}\n")

--- Starting Step 3: Load and Clean Data ---
Original train shape: (287113, 3)
Original val shape: (13368, 3)
Original test shape: (11490, 3)

Checking for null values...
Shape after dropping nulls (train): (287113, 3)

Checking for duplicate articles...
Shape after dropping duplicates (train): (284005, 3)

--- Data Loading and Cleaning Complete ---
Final training samples: 50000
Final validation samples: 5000
Final test samples: 1000



## Convert to Hugging Face

In [4]:
print("--- Starting Step 4: Convert to DatasetDict ---")

ds_train = Dataset.from_pandas(df_train)
ds_val = Dataset.from_pandas(df_val)
ds_test = Dataset.from_pandas(df_test)

raw_datasets = DatasetDict({
    'train': ds_train,
    'validation': ds_val,
    'test': ds_test
})

print(raw_datasets)
print("\n--- DatasetDict created ---\n")

--- Starting Step 4: Convert to DatasetDict ---
DatasetDict({
    train: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 1000
    })
})

--- DatasetDict created ---



## Preprocessing and Tokenization

In [5]:
print("--- Starting Step 5: Tokenization ---")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

def preprocess_function(examples):
    # Add the prefix to the articles
    inputs = [CFG.PREFIX + doc for doc in examples['article']]
    
    # Tokenize the articles (inputs)
    model_inputs = tokenizer(inputs, 
                             max_length=CFG.MAX_INPUT_LENGTH, 
                             truncation=True, 
                             padding=False) # Padding will be handled by DataCollator

    # Tokenize the summaries (labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['highlights'], 
                           max_length=CFG.MAX_TARGET_LENGTH, 
                           truncation=True,
                           padding=False)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing function to all datasets
print("Tokenizing datasets (this may take a minute)...")
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=['id', 'article', 'highlights', '__index_level_0__'] # Remove old columns
)

print(tokenized_datasets)
print("--- Tokenization complete ---\n")

--- Starting Step 5: Tokenization ---


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Tokenizing datasets (this may take a minute)...


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})
--- Tokenization complete ---



## Load Model and Data Collator

In [6]:
print("--- Starting Step 6: Load Model & Data Collator ---")

# Load the pre-trained T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(CFG.MODEL_NAME)

# Create a data collator
# This will dynamically pad the sequences in each batch
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("--- Model and Data Collator loaded ---\n")

--- Starting Step 6: Load Model & Data Collator ---


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- Model and Data Collator loaded ---



## Define Evaluation (ROUGE Metrics)

In [7]:
print("--- Starting Step 7: Define ROUGE Metric ---")

# Load the ROUGE metric
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Decode generated summaries
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Decode reference summaries
    # Replace -100 (the ignore index) with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # ROUGE-specific post-processing: add newlines
    decoded_preds = ["\n".join(pred.split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.split()) for label in decoded_labels]
    
    # Calculate ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # Extract key metrics and scale by 100
    result = {key: value * 100 for key, value in result.items()}
    
    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    # Return rounded metrics
    return {k: round(v, 4) for k, v in result.items()}

print("--- 'compute_metrics' function defined ---\n")

--- Starting Step 7: Define ROUGE Metric ---


--- 'compute_metrics' function defined ---



## Define Training Arguments and Trainer

In [8]:
from transformers import EarlyStoppingCallback

print("--- Starting Step 8: Define Trainer ---")

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=CFG.OUTPUT_DIR,
    report_to="none",                 
    
    # --- Checkpointing Strategy ---
    eval_strategy="epoch",            
    save_strategy="epoch",            
    save_total_limit=3,               
    load_best_model_at_end=True,      
    metric_for_best_model="eval_rouge2", 
    greater_is_better=True,
    # --- 'early_stopping_patience' was REMOVED from here ---
    
    # --- Progress Bar / Logging ---
    logging_strategy="steps",         
    logging_steps=100,                
    
    # --- OOM FIX: GRADIENT ACCUMULATION ---
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,   
    gradient_accumulation_steps=2,    
    
    learning_rate=CFG.LEARNING_RATE,
    weight_decay=0.01,
    num_train_epochs=CFG.NUM_EPOCHS,
    predict_with_generate=True,       
    fp16=True,                        
    push_to_hub=False
)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    
    # --- THIS IS THE CORRECT LOCATION FOR EARLY STOPPING ---
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("--- Trainer defined ---\n")

--- Starting Step 8: Define Trainer ---


/tmp/ipykernel_37/2271302429.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


--- Trainer defined ---



## Start Fine-Tuning

In [9]:
print("--- Starting Step 9: Fine-Tuning ---")
print(f"This will take a while. Training for {CFG.NUM_EPOCHS} epochs...")
train_result = trainer.train()

# Save final model, tokenizer, and training stats
trainer.save_model()
print(f"--- Fine-tuning complete. Best model saved to {CFG.OUTPUT_DIR} ---")

--- Starting Step 9: Fine-Tuning ---
This will take a while. Training for 5 epochs...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.076500,1.935665,23.491400,9.944600,18.876900,23.494600,19.993400
2,2.135300,1.935032,23.485800,9.953600,18.880100,23.493100,19.993600
3,2.144200,1.934630,23.486000,9.959600,18.885600,23.489200,19.993600
4,2.079900,1.934044,23.493100,9.969400,18.885100,23.495000,19.993400
5,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


--- Fine-tuning complete. Best model saved to t5-summarizer-with-checkpoints ---


## Evaluate on Test Set

In [10]:
print("\n--- Starting Step 10: Evaluating on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("\n--- Test Set ROUGE Scores (from best model) ---")
print(test_results)


--- Starting Step 10: Evaluating on Test Set ---



--- Test Set ROUGE Scores (from best model) ---
{'eval_loss': 1.9230390787124634, 'eval_rouge1': 24.1193, 'eval_rouge2': 10.1139, 'eval_rougeL': 19.3122, 'eval_rougeLsum': 24.1225, 'eval_gen_len': 19.992, 'eval_runtime': 167.66, 'eval_samples_per_second': 5.964, 'eval_steps_per_second': 1.491, 'epoch': 5.0}


## Example Outputs (Inference)

In [11]:
print("\n--- Starting Step 11: Generating Example Summaries ---")

# Use the fine-tuned model for summarization
summarizer_pipeline = pipeline(
    "summarization", 
    model=trainer.model,  # This is the best model
    tokenizer=tokenizer, 
    device=0
)

# Get some examples from our original test dataframe (df_test)
def show_summary(index):
    try:
        original_article = df_test.iloc[index]['article']
        reference_summary = df_test.iloc[index]['highlights']
        
        input_text = CFG.PREFIX + original_article
        
        # --- FIXED PIPELINE CALL ---
        # Use max_new_tokens, which is the preferred argument
        generated_summary = summarizer_pipeline(
            input_text, 
            max_new_tokens=CFG.MAX_TARGET_LENGTH,  # Use this instead of max_length
            min_length=30,  
            num_beams=4,    
            early_stopping=True
        )[0]['summary_text']
        # --- END FIX ---
        
        print(f"--- EXAMPLE {index + 1} ---")
        print(f"**ORIGINAL ARTICLE (truncated):**\n{original_article[:1000]}...\n")
        print(f"**REFERENCE SUMMARY:**\n{reference_summary}\n")
        print(f"**GENERATED SUMMARY:**\n{generated_summary}\n")
        print("="*50 + "\n")
    except IndexError:
        print(f"Error: Not enough test samples to show example {index + 1}")

# Show the first 3 examples
for i in range(3):
    if i < len(df_test):
        show_summary(i)

print("--- All steps complete. ---")

Device set to use cuda:0



--- Starting Step 11: Generating Example Summaries ---
--- EXAMPLE 1 ---
**ORIGINAL ARTICLE (truncated):**
A Hollywood-inspired experiment to help dementia patients by waking them up with video recordings from loved ones is taking place in New York. The hope is that the videos shown to residents of a care home in the city will ease their confusion, forgetfulness and agitation. The idea is borrowed from the 2004 Adam Sandler film 50 First Dates, in which Drew Barrymore's character suffers a brain injury and memory loss each day. A suitor, played by Sandler, uses videos to remind her of him. Charlotte Dell, director of social services at Hebrew Home in Riverdale, said the film was 'fluff', but added it inspired her to think about how the idea could help residents. The film 50 First Dates starring Adam Sandler and Drew Barrymore has inspired a new dementia experiment taking place at a care home in Riverdale, New York . 'We're looking to see if we can set a positive tone for the day,' wit